# Toxic Comment Classification

This project builds a multi-class content moderation classifier that categorizes social media comments as **normal**, **toxic**, or **severely toxic**. The dataset was originally released by the [Google Jigsaw team](https://jigsaw.google.com) as part of a public competition and underpins the [Perspective API](https://perspectiveapi.com) for hate speech detection.

The pipeline covers:
- Text featurization using Bag-of-Words (count vectorization)
- Multi-class logistic regression with cross-validated hyperparameter tuning
- Two strategies — resampling and class reweighting — to address severe class imbalance
- Model inspection to identify toxicity-associated vocabulary and its limitations

**Note:** The dataset contains offensive language. Comments are used solely to train a detection model.

## 1. Data Loading & Label Engineering

The dataset (`toxic.csv`) contains social media comments with binary labels for multiple toxicity types. For this multi-class task, comments are categorized as:
- `0` — Normal (not toxic, not severely toxic)
- `1` — Toxic
- `2` — Severely toxic (toxic AND severely_toxic both flagged)

The target is derived by summing the `toxic` and `severe_toxic` columns — a clean encoding that leverages the label structure of the dataset.

**Dataset format:**

| id | comment_text | toxic | severe_toxic | obscene | threat | insult | identity_hate |
|---|---|---|---|---|---|---|---|
| 0880df08u | I love this! | 0 | 0 | 0 | 0 | 0 | 0 |
| 08234hhf1 | You are terrible. | 1 | 0 | 0 | 0 | 0 | 0 |
| 2843jdf42 | [offensive content] | 1 | 1 | 1 | 0 | 1 | 1 |

## 2. Bag-of-Words Featurization

Text is represented using count vectorization (Bag-of-Words). Each comment becomes a vector counting word occurrences across a shared vocabulary. Key parameter choices:

| Parameter | Value | Rationale |
|-----------|-------|----------|
| `token_pattern` | English alpha only | Numbers and punctuation rarely signal toxicity |
| `min_df` | 0.00025 | Drops tokens appearing in < 0.025% of comments (likely typos/noise) |
| `binary` | False (default) | Word frequency preserved — repetition may indicate stronger toxicity |
| `max_df` | Not set | Setting it below 50% discards too much signal |

This yields a vocabulary of at most 10,000 tokens, producing a sparse feature matrix.

In [2]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer

data = pd.read_csv('toxic.csv')

X = data['comment_text'].values
y = (data['toxic'] + data['severe_toxic']).values

vectorizer = CountVectorizer(
    token_pattern=r"(?u)\b[a-zA-Z][a-zA-z]+\b",
    min_df=0.00025
)
X = vectorizer.fit_transform(X)

print(f'Feature matrix shape: {X.shape}  (comments × vocabulary tokens)')
print(f'Target array shape:   {y.shape}')
print(f'Class distribution: normal={( y==0).sum()}, toxic={(y==1).sum()}, severely toxic={(y==2).sum()}')

Feature matrix shape: (159571, 9159)  (comments × vocabulary tokens)
Target array shape:   (159571,)
Class distribution: normal=144277, toxic=13699, severely toxic=1595


The target encoding works because: normal comments have `(toxic=0, severe_toxic=0)` → sum = 0; toxic comments have `(1, 0)` → sum = 1; severely toxic have `(1, 1)` → sum = 2. The class distribution reveals the data imbalance: normal comments vastly outnumber toxic and severely toxic ones, a key challenge addressed in Section 4.

## 3. Multi-class Logistic Regression with Hyperparameter Tuning

The data is split 70/30 for training and testing. A validation curve using 5-fold cross-validation on the training set guides selection of the L2 regularization parameter `C` (larger C = weaker regularization). Five values between 0.001 and 1 are evaluated.

In [3]:
from sklearn.model_selection import train_test_split, ValidationCurveDisplay, validation_curve
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay
import numpy as np

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=2024)

model = LogisticRegression(max_iter=1000)
param_range = np.logspace(np.log10(0.001), np.log10(1), num=5)

train_scores, val_scores = validation_curve(
    model, X_train, y_train, param_name='C', param_range=param_range
)

ValidationCurveDisplay(
    param_name='C', param_range=param_range,
    train_scores=train_scores, test_scores=val_scores, score_name='Accuracy'
).plot()

# Final model with selected C
model = LogisticRegression(max_iter=1000, C=0.03)
model.fit(X_train, y_train)

pred_test = model.predict(X_test)
ConfusionMatrixDisplay.from_predictions(y_test, pred_test)

`C=0.3` is selected from the validation curve as the value achieving best validation performance without a large train/validation gap. The resulting confusion matrix reveals the class imbalance problem: while overall accuracy exceeds 90%, severely toxic comments are significantly under-detected, motivating the imbalance strategies in the next section.

## 4. Addressing Class Imbalance

Severely toxic comments are rare in the dataset, causing the model to under-predict them. Two strategies are compared:

**Resampling:** The training set is balanced by upsampling toxic and severely toxic examples (and downsampling normal ones) to equal class sizes.

**Class reweighting:** The model's loss function is adjusted via `class_weight='balanced'`, assigning higher weight to errors on rare classes during optimization. The dataset itself is unchanged.

In [ ]:
from sklearn.utils import resample

# ── Strategy 1: Resampling ──────────────────────────────────────────
X_train_df = pd.DataFrame(X_train.toarray())
y_train_df = pd.DataFrame(y_train, columns=['y'])

data_train = pd.concat([X_train_df, y_train_df], axis=1)
normal = data_train[data_train['y'] == 0]
toxic = data_train[data_train['y'] == 1]
severe_toxic = data_train[data_train['y'] == 2]

n_per_class = X_train.shape[0] // 3
balanced = pd.concat([
    resample(normal, n_samples=n_per_class),
    resample(toxic, n_samples=n_per_class),
    resample(severe_toxic, n_samples=n_per_class)
])

X_bal = balanced.drop(columns=['y'])
y_bal = balanced['y']

model_bal = LogisticRegression(max_iter=1000, C=0.03)
model_bal.fit(X_bal, y_bal)

print('Strategy 1: Resampling')
ConfusionMatrixDisplay.from_predictions(y_test, model_bal.predict(X_test))


# ── Strategy 2: Class Reweighting ───────────────────────────────────
model_rw = LogisticRegression(max_iter=1000, C=0.03, class_weight='balanced')
model_rw.fit(X_train, y_train)

print('Strategy 2: Class Reweighting')
ConfusionMatrixDisplay.from_predictions(y_test, model_rw.predict(X_test))

**Selected model: Class reweighting.**

Class reweighting better detects toxic comments while performing comparably on normal and severely toxic comments. In a content moderation context, false negatives (missed toxic content that stays live) are more harmful than false positives (normal content flagged for review). Class reweighting achieves the better recall on toxic classes without the data manipulation overhead of resampling.

## 5. Vocabulary Inspection & Model Limitations

The top 50 vocabulary tokens most strongly associated with toxicity (by model coefficient magnitude) are identified. This reveals an important limitation of BOW models: **lack of context**.

In [ ]:
import numpy as np

coef = model_rw.coef_[2]  # coefficients for severely toxic class
vocabulary = vectorizer.get_feature_names_out()

top_indices = np.argsort(coef)[-50:]
top_words = [(vocabulary[i], coef[i]) for i in top_indices]

print('Top 50 words associated with toxicity (ascending coefficient):')
for word, coef_val in top_words:
    print(f'  {word:<20} {coef_val:.4f}')

The top toxicity-associated words include many that can appear in perfectly normal contexts, words like `'son'` or `'head'`. Because the BOW model has no understanding of sentence structure or usage context, any comment containing these words risks being flagged as toxic.

This is the core limitation of BOW models for this task: **they capture statistical co-occurrence, not meaning**. A word that frequently co-occurs with toxic content will be weighted as a toxicity signal even when used innocuously. Context-aware models (e.g., transformer-based classifiers) address this limitation but require significantly more computational resources.